# Credit Agreement Analyzer - Phase 4 Validation
Testing Q&A engine against the real Ribbon Communications credit agreement.

Prerequisites: Ollama running locally with `llama3:8b` loaded.

In [1]:
import contextlib
import time
from pathlib import Path

from credit_analyzer.generation.qa_engine import QAEngine, QAResponse
from credit_analyzer.llm.factory import get_provider
from credit_analyzer.processing.chunker import Chunker, build_search_text
from credit_analyzer.processing.definitions import DefinitionsParser
from credit_analyzer.processing.pdf_extractor import PDFExtractor
from credit_analyzer.processing.section_detector import SectionDetector
from credit_analyzer.retrieval.bm25_store import BM25Store
from credit_analyzer.retrieval.embedder import Embedder
from credit_analyzer.retrieval.hybrid_retriever import HybridRetriever
from credit_analyzer.retrieval.vector_store import VectorStore

PDF_PATH = Path(r"../credit_agreement.pdf")
CHROMA_DIR = str(Path(r"../chroma_validation_p4"))

c:\Users\kbott\Projects\credit-analyzer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 0: Rebuild Pipeline (Phase 1-3)
Re-run extraction through retrieval setup.

In [2]:
# Phase 1: Extract, detect, parse, chunk
extractor = PDFExtractor()
doc = extractor.extract(PDF_PATH)

detector = SectionDetector()
sections = detector.detect_sections(doc)

defn_sections = [s for s in sections if s.section_type == "definitions"]
if not defn_sections:
    msg = "No definitions section found"
    raise RuntimeError(msg)
parser = DefinitionsParser()
defn_index = parser.parse(defn_sections[0])

chunker = Chunker()
chunks = chunker.chunk_document(sections, defn_index)
print(f"Pages: {doc.total_pages} | Sections: {len(sections)} | Terms: {len(defn_index.definitions)} | Chunks: {len(chunks)}")

Pages: 289 | Sections: 156 | Terms: 391 | Chunks: 710


In [3]:
# Phase 2: LLM provider
llm = get_provider()
print(f"Provider: {llm.model_name()}")
print(f"Available: {llm.is_available()}")

Provider: claude-sonnet-4-20250514
Available: True


In [4]:
# Phase 3: Retrieval layer
embedder = Embedder()

start = time.time()
embeddings = embedder.embed([build_search_text(c) for c in chunks])
print(f"Embedded {len(chunks)} chunks in {time.time() - start:.1f}s")

store = VectorStore(persist_directory=CHROMA_DIR)
with contextlib.suppress(Exception):
    store.delete_collection("ribbon_p4")
store.create_collection("ribbon_p4")
store.add_chunks("ribbon_p4", chunks, embeddings)

bm25 = BM25Store()
bm25.build_index(chunks)

retriever = HybridRetriever(
    vector_store=store,
    bm25_store=bm25,
    embedder=embedder,
    definitions_index=defn_index,
)
print("Retrieval layer ready")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1409.92it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedded 710 chunks in 24.0s
Retrieval layer ready


## Step 1: Basic Q&A
Test single questions with no conversation history.

In [5]:
qa = QAEngine(retriever=retriever, llm=llm)
print("QAEngine initialized")

QAEngine initialized


In [6]:
def print_response(resp: QAResponse) -> None:
    """Display a QAResponse in a readable format."""
    print(f"Answer:\n{resp.answer}\n")
    print(f"Confidence: {resp.confidence}")
    print(f"Sources ({len(resp.sources)}):")
    for s in resp.sources:
        pages = ", ".join(str(p) for p in s.page_numbers)
        print(f"  Section {s.section_id} - {s.section_title} (pp. {pages})")
        if s.relevant_text_snippet:
            print(f"    Snippet: {s.relevant_text_snippet[:120]}...")
    print(f"\nChunks retrieved: {len(resp.retrieved_chunks)}")
    print(f"LLM time: {resp.llm_response.duration_seconds:.1f}s | Tokens: {resp.llm_response.tokens_used}")
    print("-" * 80)

In [7]:
# Q1: Total revolving commitment (exact dollar amount)
qa.clear_history()
print("Q: What is the total revolving commitment amount?\n")
resp = qa.ask("What is the total revolving commitment amount?", "ribbon_p4")
print_response(resp)

Q: What is the total revolving commitment amount?

Answer:
The total revolving commitment amount is **$35,000,000**.

This is stated directly in the definition of "Total Revolving Commitments," which specifies that "the original amount of the Total Revolving Commitments is $35,000,000."

The revolving facility can be increased through the incremental facility provisions in Section 2.27, subject to the Available Incremental Amount and other conditions, but the base commitment starts at $35 million.

Confidence: HIGH
Sources (1):
  Section 2.4 - Revolving Commitments (pp. 62)
    Snippet: 2.4 Revolving Commitments. Subject to the terms and conditions hereof, each Revolving Lender severally agrees to make re...

Chunks retrieved: 5
LLM time: 3.7s | Tokens: 137
--------------------------------------------------------------------------------


In [8]:
# Q2: Financinal covenants (list them)
qa.clear_history()
print("Q: What financial covenants are included in the agreement?\n")
resp = qa.ask("What financial covenants are included in the agreement?", "ribbon_p4")
print_response(resp)

Q: What financial covenants are included in the agreement?

Answer:
Based on the provided context, the agreement includes **one financial covenant**:

## Maximum Consolidated Net Leverage Ratio (Section 7.1(a))

The borrower must maintain a Consolidated Net Leverage Ratio that does not exceed the following thresholds, tested quarterly on a trailing four-quarter basis:

• **September 2024 through December 2025**: 4.75:1.00
• **March 2026 and thereafter**: 4.00:1.00 (step-down)

The covenant becomes effective September 30, 2024.

## Equity Cure Provision (Section 7.1(b))

If the borrower fails the leverage test, it can cure the breach through a "Specified Equity Contribution" made within 10 business days after financial statements are due. Key mechanics:

• The equity contribution gets added to Consolidated Adjusted EBITDA for covenant calculation purposes
• **100% of the equity cure proceeds must immediately prepay Term Loans** (Section 7.1(b)(vii))
• No new loans can be funded during t

In [9]:
# Q3: Incremental debt capacity (your core use case)
qa.clear_history()
print("Q: How much incremental debt can the borrower incur?\n")
resp = qa.ask("How much incremental debt can the borrower incur?", "ribbon_p4")
print_response(resp)

Q: How much incremental debt can the borrower incur?

Answer:
The borrower can incur incremental debt up to the **Available Incremental Amount**, but the specific dollar amount or calculation method for this limit is not provided in the context excerpts.

Here's what we know about the incremental debt capacity:

**Types of Incremental Facilities Available:**
- **Revolving Commitment Increases** - can be requested during the Commitment Period
- **Incremental Term Facilities** - can be requested prior to the Term Loan Maturity Date, including delayed draw term loans

**Key Structural Constraint:**
After any Revolving Commitment Increase, the ratio of Total Revolving Commitments to Term Loans cannot exceed 10% (Section 2.27(a)). However, the borrower can request Incremental Facilities consisting solely of term loans.

**Leverage Test for Delayed Draw Term Loans:**
For delayed draw incremental term loans secured pari passu with existing facilities, the Consolidated First Lien Net Leverage 

In [10]:
# Q4: Restricted payments (complex multi-basket answer)
qa.clear_history()
print("Q: What restricted payments are permitted under the credit agreement?\n")
resp = qa.ask("What restricted payments are permitted under the credit agreement?", "ribbon_p4")
print_response(resp)

Q: What restricted payments are permitted under the credit agreement?

Answer:
Based on Section 7.6, the credit agreement permits several categories of restricted payments, provided no Event of Default has occurred or would result:

## Key Permitted Restricted Payments:

**1. Intercompany Distributions (Section 7.6(a))**
- Subsidiaries may make restricted payments to Loan Parties
- Non-Loan Party subsidiaries may distribute to other Group Members or equity holders on pro rata basis (or more favorable to Loan Parties)

**2. Employee Stock Purchases (Section 7.6(b))**
- Purchase of stock/options from current/former directors, officers, employees with tiered limits:
  - $2.5M annually (unused amounts carry forward)
  - Plus $5M if Consolidated Net Leverage Ratio ≤ 2.40:1.00 after the payment
  - Plus $10M if Consolidated Net Leverage Ratio ≤ 1.90:1.00 after the payment

**3. Stock-for-Stock Transactions (Section 7.6(e))**
- Purchase/redeem capital stock using proceeds from substantially c

In [11]:
# Q5: NOT FOUND test - question about something unlikely to be in the doc
qa.clear_history()
print("Q: What are the details of the borrower's pet insurance policy?\n")
resp = qa.ask("What are the details of the borrower's pet insurance policy?", "ribbon_p4")
print_response(resp)

Q: What are the details of the borrower's pet insurance policy?

Answer:
Based on the provided context, there is no information about the borrower's pet insurance policy. 

The insurance provision in Section 4.24 only contains general requirements that:
- All insurance maintained by the Loan Parties must be in full force and effect with premiums paid
- Insurance must be maintained with financially sound and reputable companies
- Coverage must be at least such amounts and against such risks as are usually insured against by similar companies in the same area

The context does not specify what types of insurance policies the borrower maintains, nor does it mention pet insurance specifically.

To find details about specific insurance policies, you would need to check:
- The insurance schedules or exhibits (if any)
- The borrower's compliance certificates or reporting requirements
- Other sections of the credit agreement that may detail specific insurance requirements

Confidence: HIGH
Sou

## Step 2: Conversation Follow-ups
Test that question reformulation works for multi-turn Q&A.

In [12]:
qa.clear_history()

# Turn 1: Establish context
print("Turn 1: What is the term loan facility size?\n")
resp1 = qa.ask("What is the term loan facility size?", "ribbon_p4")
print_response(resp1)

# Turn 2: Follow-up using "it" (requires reformulation)
print("\nTurn 2: What is the interest rate on it?\n")
resp2 = qa.ask("What is the interest rate on it?", "ribbon_p4")
print_response(resp2)

print(f"Conversation history length: {qa.history_length}")

Turn 1: What is the term loan facility size?

Answer:
Based on the provided context, I cannot determine the specific dollar amount of the term loan facility size. The excerpts from Section 2.27 discuss incremental facilities and various conditions for adding new term loans, but do not state the original size of the Term Facility.

The context shows that:
- The Borrower can request Incremental Term Facilities up to the "Available Incremental Amount" (Section 2.27(a))
- These can be increases to the existing Term Facility or new incremental term loan facilities
- Various conditions apply to incremental facilities, including leverage ratio tests and pricing restrictions

However, the "Available Incremental Amount" is not defined in the provided excerpts, nor is the original Term Facility size specified.

To find the term loan facility size, you would need to check:
- The definitions section for "Available Incremental Amount" 
- Section 2.01 or similar sections that typically specify facil

In [13]:
qa.clear_history()

# Turn 1: Ask about covenants broadly
print("Turn 1: What financial covenants does this agreement have?\n")
resp1 = qa.ask("What financial covenants does this agreement have?", "ribbon_p4")
print_response(resp1)

# Turn 2: Follow-up referencing the previous answer
print("\nTurn 2: Are there any step-downs in those levels?\n")
resp2 = qa.ask("Are there any step-downs in those levels?", "ribbon_p4")
print_response(resp2)

# Turn 3: Another follow-up
print("\nTurn 3: What happens if the borrower breaches them?\n")
resp3 = qa.ask("What happens if the borrower breaches them?", "ribbon_p4")
print_response(resp3)

print(f"Conversation history length: {qa.history_length}")

Turn 1: What financial covenants does this agreement have?

Answer:
Based on the provided context, this credit agreement contains **one financial covenant**:

## Maximum Consolidated Net Leverage Ratio (Section 7.1(a))

The borrower must maintain a Consolidated Net Leverage Ratio that does not exceed the following step-down schedule:

• **4.75:1.00** from September 30, 2024 through December 31, 2025
• **4.00:1.00** starting March 31, 2026 and thereafter

The covenant is tested quarterly based on the last day of any four consecutive trailing fiscal quarters of Holdings, with testing commencing September 30, 2024.

## Equity Cure Provision (Section 7.1(b))

If the borrower fails the leverage test, it can cure the breach through a "Specified Equity Contribution" within 10 business days after financial statements are due. Key restrictions on this cure right include:

• The equity contribution is added to Consolidated Adjusted EBITDA for covenant calculation purposes
• Must be at least two 

## Step 3: Context Assembly Inspection
Look at the raw prompt being sent to the LLM to verify structure.

In [14]:
from credit_analyzer.generation.prompts import build_context_prompt

# Manually retrieve and build context to inspect
result = retriever.retrieve(
    query="What is the total revolving commitment amount?",
    document_id="ribbon_p4",
    top_k=5,
)

prompt = build_context_prompt(
    chunks=result.chunks,
    definitions=result.injected_definitions,
    history=[],
    question="What is the total revolving commitment amount?",
)

print(f"Prompt length: {len(prompt)} chars")
print(f"Approximate tokens: ~{len(prompt) // 4}")
print("\n" + "=" * 80)
print(prompt)
print("=" * 80)

Prompt length: 4494 chars
Approximate tokens: ~1123

=== CONTEXT FROM CREDIT AGREEMENT ===

--- Source: Defined Terms (Section 1.1, Pages 9-56) ---
“Total Revolving Commitments”: at any time, the aggregate amount of the Revolving Commitments then in effect. The
original amount of the Total Revolving Commitments is $35,000,000. The L/C Commitment is a sublimit of the Total Revolving
Commitments.

--- Source: Defined Terms (Section 1.1, Pages 9-56) ---
“Revolving Commitment”: as to any Lender, the obligation of such Lender, if any, to make Revolving Loans in an aggregate
principal amount not to exceed the amount set forth under the heading “Revolving Commitment” opposite such Lender’s name on
Schedule 1.1A or in the Assignment and Assumption, Incremental Joinder, Extension Amendment or Refinancing Amendment, as
applicable, pursuant to which such Lender became a party hereto, as the same may be changed from time to time pursuant to the terms
hereof (including in connection with assignment

## Step 4: Timing & Token Budget
Verify we're within the 8K context window and check response times.

In [15]:
test_questions = [
    "What is the total revolving commitment amount?",
    "What are the financial covenant test levels?",
    "How much incremental debt can the borrower incur?",
    "Who is the administrative agent?",
    "What are the mandatory prepayment triggers?",
]

print(f"{'Question':<55} {'Time':>6} {'Tokens':>7} {'Conf':>6}")
print("-" * 80)

for q in test_questions:
    qa.clear_history()
    start = time.time()
    resp = qa.ask(q, "ribbon_p4")
    elapsed = time.time() - start
    print(
        f"{q:<55} {elapsed:>5.1f}s {resp.llm_response.tokens_used:>7} {resp.confidence:>6}"
    )

Question                                                  Time  Tokens   Conf
--------------------------------------------------------------------------------
What is the total revolving commitment amount?            3.6s     137   HIGH
What are the financial covenant test levels?              6.5s     304   HIGH
How much incremental debt can the borrower incur?         8.6s     398 MEDIUM
Who is the administrative agent?                          7.2s     326   HIGH
What are the mandatory prepayment triggers?               8.5s     428   HIGH


## Cleanup

In [16]:
store.delete_collection("ribbon_p4")
print("Cleaned up test collection")

Cleaned up test collection
